# 2. Training Model

This notebook section prepares and runs training for a Faster R-CNN model using the TensorFlow Model Garden TF2 training scripts. It checks/creates dataset splits, generates TFRecord files, updates configuration files, launches training, and exports a saved model.

## 2.1. Clean Dataset

Normalize XML annotations so downstream parsers/converters don't fail on extra whitespace or formatting differences. Delete the 'cleaned' folder to perform cleaning after the first time.

In [ ]:
import os

IMAGES_PATH = './images'
if os.path.exists(os.path.join(IMAGES_PATH, 'cleaned')):
    print("XML files have already been cleaned.")
else:
    print("Cleaning XML files...")
    !python xmlconversion.py --verbose
    print(f"Done cleaning .xml files in '{IMAGES_PATH}'.")

## 2.2. Split Dataset

Splits the dataset into training and test sets so that the testset has 10% of the images. The matching XMLs are included in the split.

In [ ]:
if os.path.exists('./images/train'):
    print("Dataset has already been split.")
else:
    print("Splitting dataset...")
    !python partition_dataset.py -x -i ./images -r 0.1
    print("Done splitting dataset.")

## 2.3. Update the .PBTXT file

Edit the label map (data/label_map.pbtxt) that maps class IDs to names; the training pipeline depends on it. Ensure num_classes in the pipeline.config matches the number of items in the pbtxt.

**File Contents:**

```json
item {
  id: 1
  name: "Lion"
}
item {
  id: 2
  name: "Ostrich"
}
item {
  id: 3
  name: "Oryx"
}

In [ ]:
path = os.path.join(os.getcwd(), 'data', 'label_map.pbtxt')
print(f"Opening {path}...")
!code "$path"

## 2.4. Create TF Records

Generates **TFRecord** files for 'train' and 'test' datasets. It checks if the specific .record file already exist; if they are missing, it executes the 'generate_tfrecord.py' script to create them from the corresponding image folders.

In [ ]:
def create_tf_record(dataset_folder_name):
    if os.path.exists(f'data/{dataset_folder_name}.record'):
        print(f"TFRecord file 'data/{dataset_folder_name}.record' already exists.")
        return
    else:
        print(f"Generating TFRecord file 'data/{dataset_folder_name}.record'...")
        command = f"python generate_tfrecord.py -x ./images/{dataset_folder_name} -l ./data/label_map.pbtxt -o ./data/{dataset_folder_name}.record"
        os.system(command)
        print(f"Done generating TFRecord file 'data/{dataset_folder_name}.record'.")

create_tf_record('train')
create_tf_record('test')

## 2.5. Update the 'pipeline.config' file

The default 'pipeline.config' file was finetuned with the following details:

| JSON Property                  | Value     | Reason                        |
|--------------------------------|-----------|------------------------------ |
| Number of classes              | 3         | Number of classes in dataset  |
| Train config batch size        | 4         | Based on machine RAM capacity |
| Train config optimizer         | Adam      | Showed stable loss decline    |
| Optimizer learning rate        | 0.0000001 | Showed stable loss decline    |

### 2.5.1. Faster R-CNN (1024x1024)

In [ ]:
PATH = os.path.join(os.getcwd(),
                    'models',
                    'TF2',
                    'faster_rcnn_resnet101_v1_1024x1024_coco17_tpu-8', 
                    'pipeline.config')
print(f"Opening configuration file at: {PATH}")
!code "$PATH"

### 2.5.2. SSD (1024x1024)

In [ ]:
import os
PATH = os.path.join(os.getcwd(),
                    'models',
                    'TF2',
                    'ssd_resnet101_v1_fpn_1024x1024_coco17_tpu-8', 
                    'pipeline.config')
print(f"Opening configuration file at: {PATH}")
!code "$PATH"

## 2.6. Train Model


The model training command prepares and executes the training of a TensorFlow 2 object detection model:
* **Path Setup**: It defines the file paths for the output model directory and the specific pipeline configuration file.
* **Execution**: It runs the 'model_main_tf2.py' script via the command line to start the training process using the calculated steps and defined paths.


### 2.6.1. Justification for Number of Epochs

**1. Translation of Epochs to Training Steps**: 
The TensorFlow Object Detection API configures training duration in terms of steps, not epochs. However, thinking in epochs is more intuitive for dataset coverage. The code calculates the total steps required to ensure the model sees the entire dataset exactly 100 times using the formula:

$$Steps = \frac{DatasetSize}{BatchSize} \times Epochs$$

With a 'BATCH_SIZE' of 4 (selected to accommodate the memory requirements of the architectures and '1024x1024' input size) and a dataset of 500 tags per class (approx. 1,500 images), this results in a substantial number of steps.

**2. Model Complexity & Convergence**: 
100 epochs was selected for the following reasons:

  * **Architecture Depth:** Deep architectures with heavy backbones generally require more iterations to fine-tune weights effectively.
  * **Gradient Noise:** Due to the small batch size (4), the gradient estimation per step is noisier. Increasing the number of epochs (and thus total steps) helps smooth out the optimization trajectory over time, ensuring the model settles into a stable minimum.
  * **Convergence Monitoring:** The actual "optimal" stopping point is determined by monitoring model convergence via TensorBoard. 100 epochs provide a sufficient runway to observe the training loss stabilize and the validation mAP (IoU) plateau. If the validation loss begins to increase while training loss decreases before 100 epochs, training can be considered complete (early stopping) to prevent overfitting.

### 2.6.2. Faster R-CNN (1024x1024)

**Delete Model Directory:** This allows the training to be restarted.

In [ ]:
!powershell Remove-Item -Path .\models\TF2\my_frcnn_1024x1024\* -Recurse -Force

**Train the model:** Run the TF2 object detection training loop


In [ ]:
TRAIN_SET_SIZE = len(os.listdir('images/train'))
NUM_EPOCHS = 100
BATCH_SIZE = 4
NUM_STEPS = int(TRAIN_SET_SIZE / BATCH_SIZE * NUM_EPOCHS)
print(f"Training set size: {TRAIN_SET_SIZE}")
print(f"Number of steps: {NUM_STEPS}")

MY_MODEL_DIR = "./models/TF2/my_frcnn_1024x1024"
PIPELINE_CONFIG_PATH = "./models/TF2/faster_rcnn_resnet101_v1_1024x1024_coco17_tpu-8/pipeline.config"

!python model_main_tf2.py \
--model_dir=$MY_MODEL_DIR \
--pipeline_config_path=$PIPELINE_CONFIG_PATH \
--num_train_steps=$NUM_STEPS \
--alsologtostderr

### 2.6.3. SSD (1024x1024)

**Delete Model Directory:** This allows the training to be restarted.

In [ ]:
!powershell Remove-Item -Path .\models\TF2\my_ssd_1024x1024\* -Recurse -Force

**Train the model:** Run the TF2 object detection training loop

In [ ]:
import os
TRAIN_SET_SIZE = len(os.listdir('images/train'))
NUM_EPOCHS = 100
BATCH_SIZE = 4
NUM_STEPS = int(TRAIN_SET_SIZE / BATCH_SIZE * NUM_EPOCHS)
print(f"Training set size: {TRAIN_SET_SIZE}")
print(f"Number of steps: {NUM_STEPS}")

MY_MODEL_DIR = "./models/TF2/my_ssd_1024x1024"
PIPELINE_CONFIG_PATH = "./models/TF2/ssd_resnet101_v1_fpn_1024x1024_coco17_tpu-8/pipeline.config"

!python model_main_tf2.py \
--model_dir=$MY_MODEL_DIR \
--pipeline_config_path=$PIPELINE_CONFIG_PATH \
--num_train_steps=$NUM_STEPS \
--alsologtostderr

## 2.7. Save Model

Export a the trained model for inference. It takes a trained model (checkpoint) and its configuration file, converts it into a deployable format accepting image tensors, and saves the result to a specified output directory.

### 2.7.1. Faster R-CNN (1024x1024)

In [ ]:
PIPELINE_CONFIG_PATH="./models/TF2/faster_rcnn_resnet101_v1_1024x1024_coco17_tpu-8/pipeline.config"
MY_MODEL_DIR="./models/TF2/my_frcnn_1024x1024"
OUTPUT_DIR="./models/TF2/my_frcnn_1024x1024/exported_model"

!python exporter_main_v2.py \
--input_type=image_tensor \
--pipeline_config_path="$PIPELINE_CONFIG_PATH" \
--trained_checkpoint_dir="$MY_MODEL_DIR" \
--output_directory="$OUTPUT_DIR"

### 2.7.2. SSD (1024x1024)

In [ ]:
PIPELINE_CONFIG_PATH="./models/TF2/ssd_resnet101_v1_fpn_1024x1024_coco17_tpu-8/pipeline.config"
MY_MODEL_DIR="./models/TF2/my_ssd_1024x1024"
OUTPUT_DIR="./models/TF2/my_ssd_1024x1024/exported_model"

!python exporter_main_v2.py \
--input_type=image_tensor \
--pipeline_config_path="$PIPELINE_CONFIG_PATH" \
--trained_checkpoint_dir="$MY_MODEL_DIR" \
--output_directory="$OUTPUT_DIR"

## 2.8. Utility Scripts

### 2.8.1. Modified 'xmlconversion.py'

Updated 'xmlconversion.py' to replace rsync with a cross-platform sync function for Windows compatibility.

``` Python
import glob
import os
import shutil

if not os.path.exists('images/cleaned'):
	os.makedirs('images/cleaned')
else:
	os.system('rm -rf images/cleaned/')
	os.makedirs('images/cleaned')

for file in glob.glob('images/*.xml'):   
	print(file)
	with open(file, 'r') as f:
		with open('images/cleaned/%s' %os.path.basename(file), 'w') as f1:
			for line in f:
				f1.write(line.rstrip().replace(" ", ""))

# os.system('rsync -a images/cleaned/ images/')

# portable replacement for: os.system('rsync -a images/cleaned/ images/')
def sync_dirs(src, dst):
    if not os.path.exists(src):
        return
    os.makedirs(dst, exist_ok=True)
    for root, dirs, files in os.walk(src):
        rel = os.path.relpath(root, src)
        target_root = os.path.join(dst, rel) if rel != '.' else dst
        os.makedirs(target_root, exist_ok=True)
        for f in files:
            shutil.copy2(os.path.join(root, f), os.path.join(target_root, f))

sync_dirs('images/cleaned', 'images')
```

### 2.8.2. Modified 'partition_dataset.py'

**NB**: A bug that made 'partition_dataset.py' to inaccurately split the dataset was fixed. The issue lies in the regular expression used to identify image files.

1. **Restrictive Character Set:** The regex ([a-zA-Z0-9\s_\\.\-\(\):])+ strictly whitelists specific characters. If your filenames contain characters not in this list (e.g., image+1.jpg, data[0].png, ~temp.jpg), they are excluded from the list, causing the total count (num_images) to be lower than the actual file count.

2. **Case Sensitivity:** The regex only looks for lowercase extensions (.jpg). It will ignore .JPG or .PNG files (common in raw datasets), further skewing the ratio.

```Python
""" usage: partition_dataset.py [-h] [-i IMAGEDIR] [-o OUTPUTDIR] [-r RATIO] [-x]

Partition dataset of images into training and testing sets

optional arguments:
  -h, --help            show this help message and exit
  -i IMAGEDIR, --imageDir IMAGEDIR
                        Path to the folder where the image dataset is stored. If not specified, the CWD will be used.
  -o OUTPUTDIR, --outputDir OUTPUTDIR
                        Path to the output folder where the train and test dirs should be created. Defaults to the same directory as IMAGEDIR.
  -r RATIO, --ratio RATIO
                        The ratio of the number of test images over the total number of images. The default is 0.1.
  -x, --xml             Set this flag if you want the xml annotation files to be processed and copied over.
"""
import os
import re
from shutil import copyfile
import argparse
import math
import random


def iterate_dir(source, dest, ratio, copy_xml):
    source = source.replace('\\', '/')
    dest = dest.replace('\\', '/')
    train_dir = os.path.join(dest, 'train')
    test_dir = os.path.join(dest, 'test')

    if not os.path.exists(train_dir):
        os.makedirs(train_dir)
    if not os.path.exists(test_dir):
        os.makedirs(test_dir)

    images = [f for f in os.listdir(source)
              if re.search(r'\.(jpg|jpeg|png)$', f, re.IGNORECASE)]

    num_images = len(images)
    num_test_images = math.ceil(ratio*num_images)

    for i in range(num_test_images):
        idx = random.randint(0, len(images)-1)
        filename = images[idx]
        copyfile(os.path.join(source, filename),
                 os.path.join(test_dir, filename))
        if copy_xml:
            xml_filename = os.path.splitext(filename)[0]+'.xml'
            copyfile(os.path.join(source, xml_filename),
                     os.path.join(test_dir,xml_filename))
        images.remove(images[idx])

    for filename in images:
        copyfile(os.path.join(source, filename),
                 os.path.join(train_dir, filename))
        if copy_xml:
            xml_filename = os.path.splitext(filename)[0]+'.xml'
            copyfile(os.path.join(source, xml_filename),
                     os.path.join(train_dir, xml_filename))


def main():

    # Initiate argument parser
    parser = argparse.ArgumentParser(description="Partition dataset of images into training and testing sets",
                                     formatter_class=argparse.RawTextHelpFormatter)
    parser.add_argument(
        '-i', '--imageDir',
        help='Path to the folder where the image dataset is stored. If not specified, the CWD will be used.',
        type=str,
        default=os.getcwd()
    )
    parser.add_argument(
        '-o', '--outputDir',
        help='Path to the output folder where the train and test dirs should be created. '
             'Defaults to the same directory as IMAGEDIR.',
        type=str,
        default=None
    )
    parser.add_argument(
        '-r', '--ratio',
        help='The ratio of the number of test images over the total number of images. The default is 0.1.',
        default=0.1,
        type=float)
    parser.add_argument(
        '-x', '--xml',
        help='Set this flag if you want the xml annotation files to be processed and copied over.',
        action='store_true'
    )
    args = parser.parse_args()

    if args.outputDir is None:
        args.outputDir = args.imageDir

    # Now we are ready to start the iteration
    iterate_dir(args.imageDir, args.outputDir, args.ratio, args.xml)


if __name__ == '__main__':
    main()
```